# SFProxy Evaluation Notebook

这个 notebook 设计成可以逐格运行。

建议流程：
1. 先填写路径和实验列表
2. 跑单个 checkpoint 的 `monotonic`
3. 跑单个 checkpoint 的 `velocity_recovery`
4. 再跑两类目前最关键的实验：
   - 不同 checkpoint / 不同数据量下的 recovery 曲线
   - 不同 sampler mixture 下的 recovery 对比

这两类实验正对应我们目前最缺的证据：
- sample 多少才够
- sample 什么才对


In [ ]:
from pathlib import Path
from datetime import datetime
import json
import subprocess

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path('/media/mengh/SharedData/zhanh/202604_midiproxy/synth-proxy')
PYTHON_BIN = 'python'

CKPT_DIR = Path('/media/mengh/SharedData/zhanh/202601_midisemi_data/synth-proxy/proxy/checkpoints/salamander_piano/piano_salamander_piano_mixed_v2_b0p3_c0p4_r0p2_s0p1_2s_default')

# 当前默认直接指向你已经训练好的主模型。
CKPT_PATH = CKPT_DIR / 'piano_salamander_piano_mixed_v2_b0p3_c0p4_r0p2_s0p1_2s_default_last_e199.ckpt'
DEVICE = 'cuda'

# 输出目录。每次运行 eval.py 时，Hydra 会在这里创建一个新的时间戳目录。
EVAL_ROOT = Path('/media/mengh/SharedData/zhanh/202601_midisemi_data/synth-proxy/proxy/eval')

# 这套 notebook 当前默认对齐 piano + 2s + default boundary。
EVAL_OVERRIDES = [
    'paths.repo_root=/media/mengh/SharedData/zhanh/202604_midiproxy/synth-proxy',
    'velocity_recovery.instrument.name=salamander_piano',
    'velocity_recovery.instrument.path=/media/mengh/SharedData/zhanh/202601_midisemi_data/soundfont/SalamanderGrandPiano/SalamanderGrandPianoV3.sfz',
    'velocity_recovery.instrument.seg_len_s=2.0',
    'velocity_recovery.instrument.nmax=64',
]

# 实验 A: 不同 checkpoint / 不同数据量下的 recovery 曲线
# 这里先用同一个 mixed_v2 训练过程里的 early / mid / late checkpoint。
SIZE_SWEEP = [
    {
        'label': 'mixed_v2_e19',
        'ckpt': CKPT_DIR / 'piano_salamander_piano_mixed_v2_b0p3_c0p4_r0p2_s0p1_2s_default_e19.ckpt',
    },
    {
        'label': 'mixed_v2_e99',
        'ckpt': CKPT_DIR / 'piano_salamander_piano_mixed_v2_b0p3_c0p4_r0p2_s0p1_2s_default_e99.ckpt',
    },
    {
        'label': 'mixed_v2_last',
        'ckpt': CKPT_PATH,
    },
]

# 实验 B: 不同 sampler mixture 下的 recovery 对比。
# 等 coverage / realism / stress / 其他 mixed recipe 的 ckpt 训练出来后，
# 直接往这里填 {'label': ..., 'ckpt': ...} 即可。留空时 notebook 会自动跳过。
MIXTURE_SWEEP = []

PROJECT_ROOT, CKPT_PATH, EVAL_ROOT


In [ ]:
def _run_eval(mode: str, ckpt_path: Path | str, extra_overrides: list[str] | None = None):
    ckpt_path = Path(ckpt_path)
    if not str(ckpt_path).endswith('.ckpt'):
        raise ValueError(f'Not a checkpoint path: {ckpt_path}')
    if not ckpt_path.exists():
        raise FileNotFoundError(ckpt_path)

    before = {p.resolve() for p in EVAL_ROOT.glob(f'{mode}_*') if p.is_dir()}
    cmd = [
        PYTHON_BIN,
        'src/eval.py',
        f'mode={mode}',
        f'ckpt_path={ckpt_path}',
        f'device={DEVICE}',
    ]
    cmd.extend(EVAL_OVERRIDES)
    if extra_overrides:
        cmd.extend(extra_overrides)

    print('Running command:')
    print(' '.join(cmd))
    completed = subprocess.run(cmd, cwd=PROJECT_ROOT, check=True, text=True)
    print('Return code:', completed.returncode)

    after = {p.resolve() for p in EVAL_ROOT.glob(f'{mode}_*') if p.is_dir()}
    new_dirs = sorted(after - before)
    latest = new_dirs[-1] if new_dirs else max(after, key=lambda p: p.stat().st_mtime)
    print('Output dir:', latest)
    return latest


def _load_velocity_result(run_dir: Path):
    result_path = Path(run_dir) / 'velocity_recovery_results.json'
    if not result_path.exists():
        raise FileNotFoundError(result_path)
    with open(result_path, 'r', encoding='utf-8') as f:
        return json.load(f), result_path


def _result_row(label: str, data: dict):
    return {
        'label': label,
        'ckpt_name': Path(data['ckpt_path']).name,
        'indomain_vel_mae': data['indomain']['vel_mae']['mean'],
        'stress_vel_mae': data['stress']['vel_mae']['mean'],
        'indomain_gain_pct': data['indomain']['mae_improvement_vs_init_pct']['mean'],
        'stress_gain_pct': data['stress']['mae_improvement_vs_init_pct']['mean'],
        'indomain_feat_mae': data['indomain']['feat_mae_at_vhat']['mean'],
        'stress_feat_mae': data['stress']['feat_mae_at_vhat']['mean'],
    }


def _plot_recovery_summary(df: pd.DataFrame, title: str):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout='constrained')

    axes[0].plot(df['label'], df['indomain_vel_mae'], marker='o', label='In-domain MAE')
    axes[0].plot(df['label'], df['stress_vel_mae'], marker='o', label='Stress MAE')
    axes[0].set_ylabel('Velocity MAE (MIDI 0-127)')
    axes[0].set_title('Recovery Error')
    axes[0].grid(alpha=0.25)
    axes[0].legend(frameon=False)

    axes[1].plot(df['label'], df['indomain_gain_pct'], marker='o', label='In-domain gain')
    axes[1].plot(df['label'], df['stress_gain_pct'], marker='o', label='Stress gain')
    axes[1].set_ylabel('MAE Improvement vs Init (%)')
    axes[1].set_title('Recovery Gain')
    axes[1].grid(alpha=0.25)
    axes[1].legend(frameon=False)

    for ax in axes:
        ax.set_xlabel('Experiment')
        ax.tick_params(axis='x', rotation=20)

    fig.suptitle(title)
    return fig


def _save_df(df: pd.DataFrame, name: str):
    out_dir = PROJECT_ROOT / 'notebook_outputs'
    out_dir.mkdir(exist_ok=True)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    out_path = out_dir / f'{name}_{timestamp}.csv'
    df.to_csv(out_path, index=False)
    print('Saved table to:', out_path)
    return out_path


def _has_experiments(items):
    return bool(items) and len(items) > 0


## 1. 单个 Checkpoint: Monotonic

先做最基础的功能性检查。

它回答的问题是：在最简单的单音条件下，proxy 对 velocity 的响应方向是否合理。

In [ ]:
monotonic_dir = _run_eval(
    mode='monotonic',
    ckpt_path=CKPT_PATH,
)
monotonic_dir


## 2. 单个 Checkpoint: Velocity Recovery

这是核心实验。它回答的问题是：proxy 提供的梯度能不能真的把 velocity 反推回来。

如果你只是想先看一版结果，这格通常是最值得先跑的。

In [ ]:
recovery_dir = _run_eval(
    mode='velocity_recovery',
    ckpt_path=CKPT_PATH,
)
recovery_dir


In [ ]:
single_result, single_result_path = _load_velocity_result(recovery_dir)
single_df = pd.DataFrame([_result_row('single_run', single_result)])
display(single_df)
single_result_path


## 3. 目前最缺的实验 A: 不同 Checkpoint / 不同数据量下的 Recovery 曲线

这一组实验直接回答：
- 模型是不是越训越可反演
- 数据变多以后，recovery 是否持续提升
- `stress` 条件是不是比 `in-domain` 更依赖数据量

你只需要先在最上面的 `SIZE_SWEEP` 里填好 `label` 和 `ckpt`。

In [ ]:
size_rows = []
size_run_dirs = []

for item in SIZE_SWEEP:
    label = item['label']
    ckpt = item['ckpt']
    print(f'\n===== Running size/checkpoint experiment: {label} =====')
    run_dir = _run_eval('velocity_recovery', ckpt)
    data, _ = _load_velocity_result(run_dir)
    size_rows.append(_result_row(label, data))
    size_run_dirs.append({'label': label, 'run_dir': str(run_dir)})

size_df = pd.DataFrame(size_rows)
display(size_df)
_save_df(size_df, 'size_or_checkpoint_sweep')


In [ ]:
_plot_recovery_summary(
    size_df,
    title='SFProxy Recovery vs Checkpoint / Data Size',
)
plt.show()


## 4. 目前最缺的实验 B: 不同 Sampler Mixture 下的 Recovery 对比

这一组实验直接回答：
- 什么样的数据组成更能让 proxy 学到稳定、可反演的 velocity response
- `coverage / realism / mixed / stress` 对 recovery 的影响分别是什么
- mixed 比例到底是真的更好，还是只是看上去更稳

你只需要先在最上面的 `MIXTURE_SWEEP` 里填好 `label` 和 `ckpt`。

In [ ]:
mix_rows = []
mix_run_dirs = []

if _has_experiments(MIXTURE_SWEEP):
    for item in MIXTURE_SWEEP:
        label = item['label']
        ckpt = item['ckpt']
        print(f'\n===== Running mixture experiment: {label} =====')
        run_dir = _run_eval('velocity_recovery', ckpt)
        data, _ = _load_velocity_result(run_dir)
        mix_rows.append(_result_row(label, data))
        mix_run_dirs.append({'label': label, 'run_dir': str(run_dir)})

mix_df = pd.DataFrame(mix_rows)
display(mix_df)
if not mix_df.empty:
    _save_df(mix_df, 'sampler_mixture_sweep')


In [ ]:
if not mix_df.empty:
    _plot_recovery_summary(
        mix_df,
        title='SFProxy Recovery vs Sampler Mixture',
    )
    plt.show()
else:
    print('MIXTURE_SWEEP is empty. Skip mixture plot for now.')


## 5. 结果整理

这一格会把两类最关键实验压成一句科研判断，方便你后续写 note 或对比实验。

In [ ]:
def summarize_df(df: pd.DataFrame, title: str):
    best_indomain = df.loc[df['indomain_vel_mae'].idxmin()]
    best_stress = df.loc[df['stress_vel_mae'].idxmin()]
    best_gain = df.loc[df['stress_gain_pct'].idxmax()]
    print(title)
    print(f"- Best in-domain MAE: {best_indomain['label']} ({best_indomain['indomain_vel_mae']:.2f})")
    print(f"- Best stress MAE: {best_stress['label']} ({best_stress['stress_vel_mae']:.2f})")
    print(f"- Best stress gain: {best_gain['label']} ({best_gain['stress_gain_pct']:.2f}%)")


summarize_df(size_df, 'Checkpoint / Data Size Sweep')
if not mix_df.empty:
    print()
    summarize_df(mix_df, 'Sampler Mixture Sweep')
else:
    print('\nSampler Mixture Sweep: skipped (MIXTURE_SWEEP is empty).')


## 6. 你最后应该拿到什么

如果这个 notebook 跑完，最关键的不是某一个单独数值，而是下面两类结论：

1. 数据规模或训练阶段的变化，会不会稳定地提升 recovery，尤其是 `stress` 条件。
2. sampler mixture 的改变，会不会明显改变 recovery 结果，从而说明哪些数据配方真正有价值。

也就是说，最后你希望得到的是：
- sample 多少才够
- sample 什么才对

这正是我们之前说“目前最缺的”那两类证据。